load the frozen results and verify their checksums

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
analysis = root / "data/analysis/09ade05f26ca8b92"
causal = root / "data/causal/87500771ea55cbd4"
output = root / "docs/images"
output.mkdir(parents=True, exist_ok=True)
completion = json.loads((analysis / "completion.json").read_text())
for name, checksum in completion["outputs"].items():
    assert hashlib.sha256((analysis / name).read_bytes()).hexdigest() == checksum, name
summary = json.loads((analysis / "summary.json").read_text())
protocol = json.loads((analysis / "protocol.json").read_text())
metrics = json.loads((analysis / "evaluation_metrics.json").read_text())
with (causal / "batch_overlap.csv").open(newline="") as source:
    overlap = list(csv.DictReader(source))
diagnosis = json.loads((causal / "causal_diagnosis.json").read_text())
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.labelcolor": "#334155", "text.color": "#14283f",
    "xtick.color": "#475569", "ytick.color": "#475569",
    "figure.facecolor": "white", "axes.facecolor": "white",
    "savefig.facecolor": "white",
})
blue, teal, orange = "#3569a8", "#168276", "#bd642a"
figure_records = {}

plot dataset allocation and average precision without implying statistical significance

In [ ]:
arms = [
    ("original", "Original data"),
    ("class_balanced", "Class balancing"),
    ("ordinary_resampling", "Ordinary resampling"),
    ("study_weighted", "Equal-study weighting"),
    ("study_resampled", "Equal-study resampling"),
    ("random_weight_control", "Random-weight control"),
]
fig, axes = plt.subplots(1, 2, figsize=(12.4, 5.4), gridspec_kw={"width_ratios": [0.9, 1.7]})
fig.suptitle("What the structural benchmark measures", x=0.055, y=0.97, ha="left", fontsize=19, weight="bold")
fig.text(0.055, 0.895, "170 fibril structures · fixed geometric features · exploratory temporal evaluation", fontsize=11, color="#64748b")
splits = ["train", "test", "reference"]
values = [summary["splits"][key]["entries"] for key in splits]
labels = ["Training", "Evaluation", "Reference groups"]
axes[0].barh(labels, values, color=[blue, teal, "#94a3b8"], height=0.52)
axes[0].set_ylim(2.65, -0.5)
axes[0].set_xlim(0, max(values) * 1.23)
axes[0].set_xlabel("Deposited conformations")
axes[0].set_title("Data allocation", loc="left", fontsize=12, weight="bold", pad=14)
for i, (key, value) in enumerate(zip(splits, values)):
    axes[0].text(value + 3, i, str(value), va="center", weight="bold")
    axes[0].text(3, i + 0.38, f'{summary["splits"][key]["groups"]} dependency groups', fontsize=9, color="#64748b")
means, seed_values = [], {}
for arm, label in arms:
    values_arm = [row["average_precision"] for row in metrics
                  if row.get("arm") == arm and row["cutoff"] == protocol["config"]["primary_cutoff"]]
    assert values_arm and all(value is not None for value in values_arm)
    means.append(float(np.mean(values_arm)))
    seed_values[arm] = values_arm
for i, ((arm, label), mean) in enumerate(zip(arms, means)):
    values_arm = seed_values[arm]
    colour = teal if arm.startswith("study_") else blue
    if len(values_arm) > 1:
        axes[1].plot(values_arm, np.full(len(values_arm), i), "o", color=colour, alpha=0.24, markersize=4)
    axes[1].plot(mean, i, "D", color=colour, markersize=7)
    axes[1].text(1.02, i, f"{mean:.3f}", va="center", fontsize=10, weight="bold")
axes[1].set_yticks(range(len(arms)), [label for _, label in arms])
axes[1].invert_yaxis()
axes[1].set_xlim(0, 1)
axes[1].set_xticks(np.linspace(0, 1, 6))
axes[1].set_xlabel("Average precision (higher is better)")
axes[1].set_title("Six descriptive baselines", loc="left", fontsize=12, weight="bold", pad=14)
axes[1].grid(axis="x", alpha=0.16)
fig.text(0.055, 0.11, "Diamonds: mean score. Faint dots: individual random seeds for stochastic arms.", fontsize=10, color="#475569")
fig.text(0.055, 0.06, "Seed spread is not a confidence interval. Equal-study resampling showed no gain here; no causal improvement is established.", fontsize=10, color="#475569")
fig.subplots_adjust(left=0.13, right=0.93, top=0.78, bottom=0.24, wspace=0.85)
path = output / "fibril_benchmark.png"
fig.savefig(path, dpi=180)
plt.close(fig)
figure_records[path.name] = {"split_counts": dict(zip(splits, values)), "mean_average_precision": dict(zip([arm for arm, _ in arms], means)),
                            "cutoff": protocol["config"]["primary_cutoff"], "seed_counts": {arm: len(values) for arm, values in seed_values.items()}}

plot preparation counts to expose the missing within-batch ph comparison

In [ ]:
overlap.sort(key=lambda row: int(row["batch_id"].rsplit("_", 1)[1]))
low = np.array([int(row["ph_7_0_samples"]) for row in overlap])
high = np.array([int(row["ph_7_4_samples"]) for row in overlap])
assert int(low.sum() + high.sum()) == diagnosis["samples"]
assert int(np.sum((low > 0) & (high > 0))) == diagnosis["supported_batches"]
fig, ax = plt.subplots(figsize=(10.5, 4.8))
fig.suptitle("Why the pH correction cannot be estimated", x=0.06, y=0.96, ha="left", fontsize=19, weight="bold")
fig.text(0.06, 0.875, "Frey et al. (2026) · coordinate-linked subset · 11 preparations across 5 batches", fontsize=11, color="#64748b")
x = np.arange(len(overlap))
for offset, counts, colour, label in [(-0.19, low, blue, "pH 7.0"), (0.19, high, orange, "pH 7.4")]:
    ax.bar(x + offset, counts, 0.34, color=colour, label=label)
    for position, count in zip(x + offset, counts):
        ax.text(position, count + 0.1, str(count), ha="center", fontsize=11, weight="bold")
ax.set_xticks(x, ["Batch " + row["batch_id"].rsplit("_", 1)[1] for row in overlap])
ax.set_ylabel("Preparations with linked coordinates")
ax.set_ylim(0, 5)
ax.set_yticks(range(6))
ax.grid(axis="y", alpha=0.15)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper right", ncol=2)
fig.text(0.06, 0.11, "No batch contains both pH groups. Multiple structures from one preparation do not provide independent comparisons.", fontsize=10, color="#475569")
fig.text(0.06, 0.055, "This blocks the proposed batch-adjusted pH analysis; it does not rule out every possible causal question.", fontsize=10, color="#475569")
fig.subplots_adjust(left=0.09, right=0.96, top=0.77, bottom=0.25)
path = output / "ph_batch_overlap.png"
fig.savefig(path, dpi=180)
plt.close(fig)
figure_records[path.name] = {"batches": overlap, "preparations": diagnosis["samples"], "supported_batches": diagnosis["supported_batches"]}

record figure inputs and plotted values so the readme images can be audited

In [ ]:
inputs = [
    analysis / "summary.json", analysis / "protocol.json", analysis / "evaluation_metrics.json",
    causal / "batch_overlap.csv", causal / "causal_diagnosis.json",
]
provenance = {
    "analysis_run": analysis.name, "causal_run": causal.name,
    "matplotlib_version": matplotlib.__version__,
    "inputs": {str(path.relative_to(root)): hashlib.sha256(path.read_bytes()).hexdigest() for path in inputs},
    "figures": {name: dict(values, sha256=hashlib.sha256((output / name).read_bytes()).hexdigest()) for name, values in figure_records.items()},
}
(output / "figure_provenance.json").write_text(json.dumps(provenance, indent=2, sort_keys=True) + "\n")
print("saved fibril_benchmark.png and ph_batch_overlap.png")